# Celeb-DF v2 Deepfake Detection — EfficientNet-B3 (Windows 11)

This notebook trains a REAL/FAKE frame classifier with **PyTorch + torchvision EfficientNet-B3** and evaluates it on the official Celeb-DF v2 test split.

- Uses pretrained EfficientNet-B3 weights.
- Uses the original Celeb-DF v2 video folders and official test-list split.
- Samples five frames from each video and keeps videos separated between train, validation, and test.
- Trains in one continuous stage (no separate fine-tuning stage).
- Uses class-weighted loss to reduce the impact of class imbalance.
- Saves the best checkpoint as `efficientnet_b3_best.pth` directly on your Windows Desktop.
- Reports frame-level and video-level metrics and plots both confusion matrices.

**No `timm` package is required.** The model comes from `torchvision.models`.

## 1. Install the additional packages

Run this cell once in the notebook's selected environment. PyTorch and torchvision should already be installed in your `cnn_env` environment. This cell does not reinstall them.

In [1]:
%pip install -q kagglehub opencv-python-headless scikit-learn matplotlib tqdm pandas pillow

Note: you may need to restart the kernel to use updated packages.


## 2. Imports, reproducibility, and Windows Desktop paths

The project folder and extracted B3 frames are kept separate from the B0/B8 frame folders so their preprocessing cannot be mixed accidentally.

In [2]:
import os
import re
import time
import random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, transforms, models
from torchvision.transforms import InterpolationMode
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)
from tqdm.auto import tqdm
from pathlib import Path
from contextlib import nullcontext

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if os.name == "nt":
    try:
        import winreg
        with winreg.OpenKey(
            winreg.HKEY_CURRENT_USER,
            r"Software\Microsoft\Windows\CurrentVersion\Explorer\User Shell Folders"
        ) as key:
            desktop_value, _ = winreg.QueryValueEx(key, "Desktop")
        DESKTOP_DIR = os.path.expandvars(desktop_value)
    except (ImportError, FileNotFoundError, OSError):
        DESKTOP_DIR = os.path.join(os.path.expanduser("~"), "Desktop")
else:
    DESKTOP_DIR = os.path.join(os.path.expanduser("~"), "Desktop")

os.makedirs(DESKTOP_DIR, exist_ok=True)
PROJECT_DIR = os.path.join(DESKTOP_DIR, "Deepfake_Project")
FRAME_ROOT = os.path.join(PROJECT_DIR, "celebdf_frames_b3")
MANIFEST_DIR = os.path.join(PROJECT_DIR, "manifests_b3")
BEST_MODEL_PATH = os.path.join(DESKTOP_DIR, "efficientnet_b3_best.pth")
METRICS_PATH = os.path.join(DESKTOP_DIR, "efficientnet_b3_test_metrics.csv")

for folder in [PROJECT_DIR, FRAME_ROOT, MANIFEST_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Python:", os.sys.executable)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Desktop:", DESKTOP_DIR)
print("Frames:", FRAME_ROOT)
print("Best model will be saved to:", BEST_MODEL_PATH)

Python: c:\WINDOWS\system32\cnn_env\Scripts\python.exe
PyTorch: 2.11.0+cu128
Torchvision: 0.26.0+cu128
Desktop: C:\Users\abdul\OneDrive\Desktop
Frames: C:\Users\abdul\OneDrive\Desktop\Deepfake_Project\celebdf_frames_b3
Best model will be saved to: C:\Users\abdul\OneDrive\Desktop\efficientnet_b3_best.pth


## 3. Download Celeb-DF v2

KaggleHub downloads the dataset to its local cache. The notebook uses that downloaded folder directly.

In [3]:
import kagglehub

dataset_path = kagglehub.dataset_download("reubensuju/celeb-df-v2")
print("Dataset downloaded to:", dataset_path)

print("\nTop-level contents:")
for item in sorted(os.listdir(dataset_path)):
    full_path = os.path.join(dataset_path, item)
    print(item, "—", "folder" if os.path.isdir(full_path) else "file")

Dataset downloaded to: C:\Users\abdul\.cache\kagglehub\datasets\reubensuju\celeb-df-v2\versions\1

Top-level contents:
Celeb-real — folder
Celeb-synthesis — folder
List_of_testing_videos.txt — file
YouTube-real — folder


## 4. Build the video dataframe

The video dataframe uses `REAL = 0` and `FAKE = 1` for stratification and reporting. Later, `ImageFolder` assigns its own alphabetical class order: `FAKE = 0`, `REAL = 1`. The notebook checks and stores that mapping in the checkpoint.

In [4]:
real_folders = ["YouTube-real", "Celeb-real"]
fake_folders = ["Celeb-synthesis"]
VIDEO_EXTENSIONS = (".mp4", ".avi", ".mov")
records = []

for folder in real_folders:
    folder_path = os.path.join(dataset_path, folder)
    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f"Required folder not found: {folder_path}")
    for filename in sorted(os.listdir(folder_path)):
        if filename.lower().endswith(VIDEO_EXTENSIONS):
            records.append({
                "path": os.path.join(folder_path, filename),
                "video": filename,
                "label": 0,
                "class": "REAL"
            })

for folder in fake_folders:
    folder_path = os.path.join(dataset_path, folder)
    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f"Required folder not found: {folder_path}")
    for filename in sorted(os.listdir(folder_path)):
        if filename.lower().endswith(VIDEO_EXTENSIONS):
            records.append({
                "path": os.path.join(folder_path, filename),
                "video": filename,
                "label": 1,
                "class": "FAKE"
            })

df = pd.DataFrame(records)
if df.empty:
    raise ValueError("No videos were found. Check the downloaded dataset structure.")

print("Total videos:", len(df))
print("\nVideo counts by class:")
print(df["class"].value_counts())
display(df.head())

Total videos: 6529

Video counts by class:
class
FAKE    5639
REAL     890
Name: count, dtype: int64


,path,video,label,class
0,C:\Users\abdul\.cache\kagglehub\datasets\reube...,00000.mp4,0,REAL
1,C:\Users\abdul\.cache\kagglehub\datasets\reube...,00001.mp4,0,REAL
2,C:\Users\abdul\.cache\kagglehub\datasets\reube...,00002.mp4,0,REAL
3,C:\Users\abdul\.cache\kagglehub\datasets\reube...,00003.mp4,0,REAL
4,C:\Users\abdul\.cache\kagglehub\datasets\reube...,00004.mp4,0,REAL


## 5. Create train, validation, and official test splits

The official Celeb-DF test list is reserved for final evaluation. Only the remaining videos are split into train and validation sets. A video's frames therefore cannot leak across these splits.

In [5]:
test_list_path = os.path.join(dataset_path, "List_of_testing_videos.txt")
if not os.path.isfile(test_list_path):
    raise FileNotFoundError(f"Official test list not found: {test_list_path}")

with open(test_list_path, "r", encoding="utf-8") as file:
    test_relpaths = set()
    for line in file:
        parts = line.strip().split(maxsplit=1)
        if len(parts) == 2:
            test_relpaths.add(parts[1].replace("\\", "/").casefold())

df["relative_path"] = df["path"].apply(
    lambda path: os.path.relpath(path, dataset_path).replace("\\", "/")
)
df["split"] = "remaining"
df.loc[df["relative_path"].str.casefold().isin(test_relpaths), "split"] = "test"

matched = int((df["split"] == "test").sum())
print("Official test videos matched:", matched)
print("Test-list entries:", len(test_relpaths))
if matched == 0:
    raise ValueError(
        "No official test-list entries matched the discovered videos. "
        "Check the dataset folder structure and test-list path format."
    )

remaining = df[df["split"] == "remaining"].copy()
train_df, val_df = train_test_split(
    remaining,
    test_size=0.15,
    random_state=SEED,
    stratify=remaining["label"]
)
train_df = train_df.copy()
val_df = val_df.copy()
test_df = df[df["split"] == "test"].copy()
train_df["split"] = "train"
val_df["split"] = "val"

for split_name, split_data in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Official test", test_df)
]:
    print(f"\n{split_name}: {len(split_data)} videos")
    print(split_data["class"].value_counts())

train_df.to_csv(os.path.join(MANIFEST_DIR, "train_videos.csv"), index=False)
val_df.to_csv(os.path.join(MANIFEST_DIR, "val_videos.csv"), index=False)
test_df.to_csv(os.path.join(MANIFEST_DIR, "test_videos.csv"), index=False)
print("\nSplit manifests saved in:", MANIFEST_DIR)

Official test videos matched: 518
Test-list entries: 518

Train: 5109 videos
class
FAKE    4504
REAL     605
Name: count, dtype: int64

Validation: 902 videos
class
FAKE    795
REAL    107
Name: count, dtype: int64

Official test: 518 videos
class
FAKE    340
REAL    178
Name: count, dtype: int64

Split manifests saved in: C:\Users\abdul\OneDrive\Desktop\Deepfake_Project\manifests_b3


## 6. Extract five frames per video

Five frames are sampled evenly across each video. Frames are stored in separate FAKE/REAL folders for each split. The B3 extraction folder is independent of the B0 and B8 folders. Frames are saved with aspect ratio preserved and a maximum side length of 512 pixels; the model transforms them to its 300×300 input size during training and evaluation.

In [6]:
FRAMES_PER_VIDEO = 5
MAX_FRAME_SIDE = 512

for split_name in ["train", "val", "test"]:
    for class_name in ["FAKE", "REAL"]:
        os.makedirs(os.path.join(FRAME_ROOT, split_name, class_name), exist_ok=True)

for split_name, split_data in [
    ("train", train_df),
    ("val", val_df),
    ("test", test_df)
]:
    for _, row in tqdm(
        split_data.iterrows(),
        total=len(split_data),
        desc=f"Extracting {split_name} frames"
    ):
        video_path = row["path"]
        class_name = row["class"]
        output_dir = os.path.join(FRAME_ROOT, split_name, class_name)
        relative_video_path = os.path.relpath(video_path, dataset_path)
        video_id = os.path.splitext(relative_video_path)[0]
        video_id = re.sub(r"[^A-Za-z0-9._-]+", "_", video_id.replace("\\", "_"))

        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print("Could not open video:", video_path)
            cap.release()
            continue

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0:
            print("No readable frames:", video_path)
            cap.release()
            continue

        frame_indices = np.linspace(
            0,
            total_frames - 1,
            min(FRAMES_PER_VIDEO, total_frames),
            dtype=int
        )
        expected_paths = [
            os.path.join(output_dir, f"{video_id}_frame_{i}.jpg")
            for i in range(len(frame_indices))
        ]

        if all(os.path.isfile(path) for path in expected_paths):
            cap.release()
            continue

        for frame_number, frame_index in enumerate(frame_indices):
            output_path = os.path.join(output_dir, f"{video_id}_frame_{frame_number}.jpg")
            if os.path.isfile(output_path):
                continue

            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
            success, frame = cap.read()
            if not success:
                continue

            height, width = frame.shape[:2]
            largest_side = max(height, width)
            if largest_side > MAX_FRAME_SIDE:
                scale = MAX_FRAME_SIDE / largest_side
                new_width = max(1, int(width * scale))
                new_height = max(1, int(height * scale))
                frame = cv2.resize(
                    frame,
                    (new_width, new_height),
                    interpolation=cv2.INTER_AREA
                )

            cv2.imwrite(
                output_path,
                frame,
                [cv2.IMWRITE_JPEG_QUALITY, 95]
            )

        cap.release()

print("Frame extraction complete!")

Extracting train frames:   0%|          | 0/5109 [00:00<?, ?it/s]

Extracting val frames:   0%|          | 0/902 [00:00<?, ?it/s]

Extracting test frames:   0%|          | 0/518 [00:00<?, ?it/s]

Frame extraction complete!


## 7. Verify the extracted frames

In [7]:
frame_counts = []
for split_name in ["train", "val", "test"]:
    for class_name in ["FAKE", "REAL"]:
        folder = os.path.join(FRAME_ROOT, split_name, class_name)
        count = len([
            name for name in os.listdir(folder)
            if name.lower().endswith(".jpg")
        ]) if os.path.isdir(folder) else 0
        frame_counts.append({"split": split_name, "class": class_name, "frames": count})
        print(f"{split_name:5} | {class_name:4} | {count} frames")

if any(row["frames"] == 0 for row in frame_counts):
    raise ValueError("At least one split/class folder has no frames. Check extraction and dataset paths.")

train | FAKE | 22520 frames
train | REAL | 3021 frames
val   | FAKE | 3975 frames
val   | REAL | 535 frames
test  | FAKE | 1700 frames
test  | REAL | 890 frames


## 8. Create datasets and data loaders

EfficientNet-B3's standard pretrained input crop is 300×300. ImageNet normalization is used. Class weights are computed from the training-frame counts; this gives the underrepresented class more influence in the loss without changing the validation or official test distributions.

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type != "cuda":
    print("WARNING: Training will run on CPU and may be slow. A CUDA-enabled NVIDIA GPU is recommended.")

MODEL_NAME = "efficientnet_b3"
IMAGE_SIZE = 300
MODEL_MEAN = (0.485, 0.456, 0.406)
MODEL_STD = (0.229, 0.224, 0.225)
NUM_CLASSES = 2

train_path = os.path.join(FRAME_ROOT, "train")
val_path = os.path.join(FRAME_ROOT, "val")
test_path = os.path.join(FRAME_ROOT, "test")

train_transform = transforms.Compose([
    transforms.Resize(320, interpolation=InterpolationMode.BICUBIC),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=MODEL_MEAN, std=MODEL_STD)
])

eval_transform = transforms.Compose([
    transforms.Resize(320, interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=MODEL_MEAN, std=MODEL_STD)
])

train_ds = datasets.ImageFolder(train_path, transform=train_transform)
val_ds = datasets.ImageFolder(val_path, transform=eval_transform)

print("Class mapping:", train_ds.class_to_idx)
print("Training frames:", len(train_ds))
print("Validation frames:", len(val_ds))

if len(train_ds) == 0 or len(val_ds) == 0:
    raise ValueError("Training or validation dataset is empty. Check the extracted frame folders.")
if train_ds.class_to_idx != val_ds.class_to_idx:
    raise ValueError("Train and validation class mappings do not match.")

class_counts = np.bincount(train_ds.targets, minlength=NUM_CLASSES)
if np.any(class_counts == 0):
    raise ValueError("At least one class has no training frames.")

class_weights = len(train_ds) / (NUM_CLASSES * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)
print("Class counts in ImageFolder order:", class_counts.tolist())
print("Class weights:", class_weights.tolist())

BATCH_SIZE = 8
NUM_WORKERS = 0

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Using device: cuda
Class mapping: {'FAKE': 0, 'REAL': 1}
Training frames: 25541
Validation frames: 4510
Class counts in ImageFolder order: [22520, 3021]
Class weights: [0.5670737028121948, 4.227242469787598]
Training batches: 3193
Validation batches: 564


## 9. Load pretrained EfficientNet-B3

The classifier's final layer is replaced with a two-output layer for REAL and FAKE. All model parameters are trainable from the first training epoch; there is no separate freeze/unfreeze or fine-tuning phase.

In [9]:
weights = models.EfficientNet_B3_Weights.DEFAULT
model = models.efficientnet_b3(weights=weights)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
model = model.to(device)

print("Model:", MODEL_NAME)
print("Input size:", IMAGE_SIZE, "x", IMAGE_SIZE)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Pretrained weights:", weights)

Model: efficientnet_b3
Input size: 300 x 300
Trainable parameters: 10699306
Pretrained weights: EfficientNet_B3_Weights.IMAGENET1K_V1


## 10. Train EfficientNet-B3

The best checkpoint is selected using validation macro F1, rather than accuracy alone. Mixed precision is enabled automatically when a CUDA GPU is available. If you get a CUDA out-of-memory error, lower `BATCH_SIZE` in Section 8 to `4` or `2`, then rerun Sections 8–10.

In [10]:
EPOCHS = 15
PATIENCE = 4
ACCUMULATION_STEPS = 2
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
USE_AMP = device.type == "cuda"

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)
try:
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


def autocast_context():
    if not USE_AMP:
        return nullcontext()
    try:
        return torch.amp.autocast(device_type="cuda", dtype=torch.float16)
    except (AttributeError, TypeError):
        return torch.cuda.amp.autocast()

best_f1 = -1.0
early_stop_count = 0
history = []

print("Training configuration")
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Gradient accumulation:", ACCUMULATION_STEPS)
print("Effective batch size:", BATCH_SIZE * ACCUMULATION_STEPS)
print("Mixed precision:", USE_AMP)
print("Checkpoint path:", BEST_MODEL_PATH)

for epoch in range(EPOCHS):
    epoch_start = time.time()
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    model.train()
    optimizer.zero_grad(set_to_none=True)
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        group_start = (batch_idx // ACCUMULATION_STEPS) * ACCUMULATION_STEPS
        current_group_size = min(
            ACCUMULATION_STEPS,
            len(train_loader) - group_start
        )

        with autocast_context():
            outputs = model(images)
            raw_loss = criterion(outputs, labels)
            loss = raw_loss / current_group_size

        scaler.scale(loss).backward()

        if (batch_idx + 1) % ACCUMULATION_STEPS == 0 or batch_idx + 1 == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        batch_size_now = labels.size(0)
        train_loss_sum += raw_loss.item() * batch_size_now
        train_correct += (outputs.argmax(dim=1) == labels).sum().item()
        train_total += batch_size_now

        if (batch_idx + 1) % 100 == 0 or batch_idx + 1 == len(train_loader):
            print(
                f"Train [{batch_idx + 1}/{len(train_loader)}] "
                f"Loss: {raw_loss.item():.4f} "
                f"Accuracy: {100 * train_correct / train_total:.2f}%"
            )

    train_loss = train_loss_sum / max(1, train_total)
    train_accuracy = train_correct / max(1, train_total)

    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0
    val_targets = []
    val_predictions = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with autocast_context():
                outputs = model(images)
                loss = criterion(outputs, labels)

            predictions = outputs.argmax(dim=1)
            val_loss_sum += loss.item() * labels.size(0)
            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)
            val_targets.extend(labels.cpu().numpy().tolist())
            val_predictions.extend(predictions.cpu().numpy().tolist())

    val_loss = val_loss_sum / max(1, val_total)
    val_accuracy = val_correct / max(1, val_total)
    val_macro_f1 = f1_score(val_targets, val_predictions, average="macro", zero_division=0)
    scheduler.step(val_macro_f1)

    epoch_time = (time.time() - epoch_start) / 60
    print(f"Train loss: {train_loss:.4f} | Train accuracy: {train_accuracy:.4f}")
    print(f"Val loss:   {val_loss:.4f} | Val accuracy:   {val_accuracy:.4f}")
    print(f"Val macro F1: {val_macro_f1:.4f} | Epoch time: {epoch_time:.2f} min")

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_accuracy": train_accuracy,
        "val_accuracy": val_accuracy,
        "val_macro_f1": val_macro_f1
    })

    if val_macro_f1 > best_f1:
        best_f1 = val_macro_f1
        early_stop_count = 0
        checkpoint = {
            "model_state_dict": model.state_dict(),
            "architecture": MODEL_NAME,
            "num_classes": NUM_CLASSES,
            "class_to_idx": train_ds.class_to_idx,
            "input_size": IMAGE_SIZE,
            "mean": MODEL_MEAN,
            "std": MODEL_STD,
            "epoch": epoch + 1,
            "best_val_macro_f1": best_f1,
            "best_val_loss": val_loss,
            "best_val_accuracy": val_accuracy,
            "torch_version": torch.__version__,
            "torchvision_version": torchvision.__version__
        }
        torch.save(checkpoint, BEST_MODEL_PATH)
        print("Best model saved:", BEST_MODEL_PATH)
    else:
        early_stop_count += 1

    if early_stop_count >= PATIENCE:
        print("Early stopping: validation macro F1 did not improve.")
        break

history_df = pd.DataFrame(history)
print("\nTraining complete.")
print("Best validation macro F1:", round(best_f1, 4))
print("Best model:", BEST_MODEL_PATH)

Training configuration
Epochs: 15
Batch size: 8
Gradient accumulation: 2
Effective batch size: 16
Mixed precision: True
Checkpoint path: C:\Users\abdul\OneDrive\Desktop\efficientnet_b3_best.pth

Epoch 1/15
Train [100/3193] Loss: 0.7988 Accuracy: 81.88%
Train [200/3193] Loss: 0.6303 Accuracy: 85.00%
Train [300/3193] Loss: 0.4905 Accuracy: 84.08%
Train [400/3193] Loss: 0.3720 Accuracy: 83.91%


KeyboardInterrupt: 

## 11. Plot training and validation curves

In [ ]:
if not history_df.empty:
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("EfficientNet-B3 Loss")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_accuracy"], marker="o", label="Train accuracy")
    plt.plot(history_df["epoch"], history_df["val_accuracy"], marker="o", label="Validation accuracy")
    plt.plot(history_df["epoch"], history_df["val_macro_f1"], marker="o", label="Validation macro F1")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title("EfficientNet-B3 Training Metrics")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

NameError: name 'history_df' is not defined

## 12. Evaluate the best checkpoint on the official test set

This section reports **frame-level** metrics and **video-level** metrics. For video-level evaluation, the FAKE probabilities of the sampled frames from each video are averaged, and the video is classified using a 0.5 threshold. The official test videos are not used during training or checkpoint selection.

In [ ]:
def load_checkpoint(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)

if not os.path.isfile(BEST_MODEL_PATH):
    raise FileNotFoundError("Best checkpoint not found. Run the training section first.")

checkpoint = load_checkpoint(BEST_MODEL_PATH, device)
if checkpoint["architecture"] != MODEL_NAME:
    raise ValueError("The checkpoint architecture does not match this notebook.")

model = models.efficientnet_b3(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

saved_size = int(checkpoint["input_size"])
saved_mean = tuple(checkpoint["mean"])
saved_std = tuple(checkpoint["std"])

final_test_transform = transforms.Compose([
    transforms.Resize(320, interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(saved_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=saved_mean, std=saved_std)
])

test_ds = datasets.ImageFolder(test_path, transform=final_test_transform)
if test_ds.class_to_idx != checkpoint["class_to_idx"]:
    raise ValueError(
        f"Test class mapping {test_ds.class_to_idx} differs from saved mapping "
        f"{checkpoint['class_to_idx']}"
    )
if len(test_ds) == 0:
    raise ValueError("Official test frame dataset is empty.")

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)

class_names = [
    name for name, index in sorted(test_ds.class_to_idx.items(), key=lambda item: item[1])
]
class_to_idx = test_ds.class_to_idx
fake_index = class_to_idx["FAKE"]
real_index = class_to_idx["REAL"]

frame_true = []
frame_pred = []
frame_fake_probs = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Evaluating official test frames"):
        images = images.to(device, non_blocking=True)
        with autocast_context():
            logits = model(images)
        probabilities = torch.softmax(logits, dim=1)
        predictions = logits.argmax(dim=1)

        frame_true.extend(labels.numpy().tolist())
        frame_pred.extend(predictions.cpu().numpy().tolist())
        frame_fake_probs.extend(probabilities[:, fake_index].cpu().numpy().tolist())

print("Class order:", class_names)
print("Test frames:", len(frame_true))


def summarize_metrics(y_true, y_pred, fake_probabilities, label="Evaluation"):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    fake_probabilities = np.asarray(fake_probabilities)
    y_true_fake = (y_true == fake_index).astype(int)

    results = {
        "evaluation": label,
        "samples": int(len(y_true)),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0)
    }
    try:
        results["roc_auc_fake"] = roc_auc_score(y_true_fake, fake_probabilities)
    except ValueError:
        results["roc_auc_fake"] = np.nan

    print(f"\n{label} classification report:")
    print(classification_report(
        y_true,
        y_pred,
        labels=list(range(len(class_names))),
        target_names=class_names,
        zero_division=0,
        digits=4
    ))
    print(f"{label} summary:")
    for key, value in results.items():
        if key != "evaluation":
            print(f"{key}: {value}")
    return results


def save_confusion_matrix(y_true, y_pred, title, output_path):
    matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(len(class_names)))
    )
    display_matrix = ConfusionMatrixDisplay(
        confusion_matrix=matrix,
        display_labels=class_names
    )
    fig, ax = plt.subplots(figsize=(6, 5))
    display_matrix.plot(
        ax=ax,
        cmap="Blues",
        values_format="d",
        colorbar=False
    )
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Confusion matrix saved to:", output_path)
    return matrix

frame_metrics = summarize_metrics(
    frame_true,
    frame_pred,
    frame_fake_probs,
    label="Frame-level test"
)
frame_cm_path = os.path.join(DESKTOP_DIR, "efficientnet_b3_frame_confusion_matrix.png")
frame_cm = save_confusion_matrix(
    frame_true,
    frame_pred,
    "EfficientNet-B3 — Frame-level Confusion Matrix",
    frame_cm_path
)

video_groups = {}
for (image_path, actual_label), fake_probability in zip(test_ds.samples, frame_fake_probs):
    frame_stem = Path(image_path).stem
    video_id = frame_stem.rsplit("_frame_", 1)[0]
    if video_id not in video_groups:
        video_groups[video_id] = {"label": actual_label, "fake_probs": []}
    video_groups[video_id]["fake_probs"].append(float(fake_probability))

video_true = []
video_fake_probs = []
video_pred = []
for video_id, values in sorted(video_groups.items()):
    mean_fake_probability = float(np.mean(values["fake_probs"]))
    predicted_label = fake_index if mean_fake_probability >= 0.5 else real_index
    video_true.append(int(values["label"]))
    video_fake_probs.append(mean_fake_probability)
    video_pred.append(predicted_label)

video_metrics = summarize_metrics(
    video_true,
    video_pred,
    video_fake_probs,
    label="Video-level test"
)
video_cm_path = os.path.join(DESKTOP_DIR, "efficientnet_b3_video_confusion_matrix.png")
video_cm = save_confusion_matrix(
    video_true,
    video_pred,
    "EfficientNet-B3 — Video-level Confusion Matrix",
    video_cm_path
)

metrics_df = pd.DataFrame([frame_metrics, video_metrics])
metrics_df.to_csv(METRICS_PATH, index=False)
print("\nEvaluation metrics saved to:", METRICS_PATH)
print("Video predictions were aggregated from", len(video_groups), "test videos.")

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL torch.torch_version.TorchVersion was not an allowed global by default. Please use `torch.serialization.add_safe_globals([torch.torch_version.TorchVersion])` or the `torch.serialization.safe_globals([torch.torch_version.TorchVersion])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load 

## 13. Predict REAL or FAKE for a separate video (optional)

Set `VIDEO_PATH` to a video on your computer and run this cell. It samples frames throughout the video and averages their FAKE probabilities. This is an inference demonstration, not part of the official test evaluation.

In [ ]:
from PIL import Image

inference_transform = transforms.Compose([
    transforms.Resize(320, interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(saved_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=saved_mean, std=saved_std)
])

def predict_video(video_path, num_frames=20):
    if not os.path.isfile(video_path):
        raise FileNotFoundError(f"Video not found: {video_path}")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        cap.release()
        raise ValueError("Could not open the selected video.")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        raise ValueError("The selected video has no readable frames.")

    frame_indices = np.linspace(
        0,
        total_frames - 1,
        min(num_frames, total_frames),
        dtype=int
    )
    fake_probabilities = []

    for frame_index in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        success, frame = cap.read()
        if not success:
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(frame)
        image_tensor = inference_transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            with autocast_context():
                logits = model(image_tensor)
            fake_probability = torch.softmax(logits, dim=1)[0, fake_index].item()
        fake_probabilities.append(fake_probability)

    cap.release()
    if not fake_probabilities:
        raise ValueError("No readable frames were extracted from the video.")

    average_fake = float(np.mean(fake_probabilities))
    prediction = "FAKE" if average_fake >= 0.5 else "REAL"
    print("Frames analyzed:", len(fake_probabilities))
    print(f"Average FAKE probability: {average_fake:.2%}")
    print(f"Average REAL probability: {1 - average_fake:.2%}")
    print("Prediction:", prediction)
    return prediction, average_fake

VIDEO_PATH = r""

if VIDEO_PATH.strip():
    predict_video(VIDEO_PATH.strip().strip('"'))
else:
    print("Set VIDEO_PATH to a local video path to run a prediction.")

NameError: name 'saved_size' is not defined

## Output files

The notebook saves these files directly on your Windows Desktop (including a Desktop folder redirected to OneDrive):

- `efficientnet_b3_best.pth` — best checkpoint selected using validation macro F1.
- `efficientnet_b3_test_metrics.csv` — frame-level and video-level test metrics.
- `efficientnet_b3_frame_confusion_matrix.png` — frame-level confusion matrix.
- `efficientnet_b3_video_confusion_matrix.png` — video-level confusion matrix.

Extracted frames and split CSV files are stored in `Desktop\Deepfake_Project\celebdf_frames_b3` and `Desktop\Deepfake_Project\manifests_b3`.

**Evaluation note:** The confusion matrices and metrics are calculated on the official test split. Frame-level results count each sampled frame; video-level results average the sampled frames' FAKE probabilities for each video. Video-level metrics are the more direct estimate of performance when the final application makes one prediction per video.